# 에피소딕 메모리를 활용한 멀티 에이전트 헬스케어 시스템

## 소개

이 노트북은 AgentCore 메모리 SDK와 Strands 메모리 훅을 사용하여 **에피소딕 메모리가 포함된 멀티 에이전트 헬스케어 시스템**을 구현하는 방법을 보여줍니다. 이 접근 방식은 수동 API 호출 없이 자동 메모리 관리를 제공합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:----------------------------------------------------------------------------------|
| 튜토리얼 유형       | 멀티 에이전트 조율을 활용한 에피소딕 메모리                                      |
| 에이전트 유형       | 헬스케어 어시스턴트 시스템                                                       |
| 에이전트 프레임워크 | 메모리 훅이 포함된 Strands 에이전트                                              |
| LLM 모델            | Anthropic Claude Sonnet 4                                                        |
| 튜토리얼 구성 요소  | 에피소딕 메모리, 메모리 훅, HealthLake 통합                                     |
| 난이도              | 중급                                                                             |

학습 내용:

- MemoryClient SDK를 사용한 에피소딕 메모리 활용법
- 자동 메모리 관리를 위한 메모리 훅 생성
- 공유 에피소딕 메모리를 사용하는 전문 에이전트 구현
- 실시간 HealthLake FHIR 쿼리 통합

## 에피소딕 메모리가 이 헬스케어 어시스턴트에 도움이 되는 방법

**EpisodicStrategy**는 상호작용을 구조화된 에피소드로 캡처하고 세션 간에 의미 있는 인사이트를 생성합니다. 이는 "무엇이 일어났는지"를 기록하는 것을 넘어 상호작용이 "왜" 그리고 "어떻게" 전개되었는지를 이해합니다.

### 3단계 프로세스

1. **추출(Extraction)** - 단기 메모리(이벤트)에서 유용한 인사이트를 식별하고 구조화된 에피소드로 장기 메모리에 배치
2. **통합(Consolidation)** - 정보를 새 에피소드에 기록할지 기존 에피소드를 업데이트할지 결정
3. **반영(Reflection)** - 여러 에피소드에 걸쳐 인사이트를 생성하여 패턴과 개선점 식별

### 에피소드 구조

각 에피소드는 다음을 캡처합니다:
- **상황(Situation)**: 헬스케어 전문가가 달성하려는 목표
- **의도(Intent)**: 상호작용의 주요 목표
- **평가(Assessment)**: 목표가 성공적으로 달성되었는지 여부
- **근거(Justification)**: 평가가 내려진 이유
- **턴별 분석**: 에이전트 라우팅, 도구 사용 및 의사결정을 보여주는 상세 분석
- **에피소드 수준 반영**: 이 특정 세션에서 잘 작동한 것에 대한 인사이트

### 환자 수준 반영

반영은 여러 에피소드를 통합하여 더 넓은 인사이트를 추출합니다:
- **성공적인 전략**: 일관되게 작동하는 패턴 (예: 라우팅 프로토콜, 데이터 표시)
- **일반적인 사용 사례**: 이 환자에 대해 자주 수행되는 문의 유형
- **잠재적 개선 사항**: 어시스턴트가 더 효과적일 수 있는 영역
- **학습된 교훈**: 여러 상호작용에 걸친 인사이트

### 헬스케어 워크플로우의 이점

1. **라우팅 개선**: 어떤 에이전트가 어떤 유형의 질문을 가장 효과적으로 처리하는지 학습
2. **데이터 표시 개선**: 빠른 이해를 위해 복잡한 헬스케어 데이터를 포맷하는 방법 이해
3. **패턴 인식**: 특정 환자에 대한 일반적인 문의 패턴 식별
4. **품질 개선**: 여러 세션에서 무엇이 효과적이고 무엇이 그렇지 않은지 추적
5. **문맥 인식**: 미래 상호작용이 과거 세션에서 학습된 교훈의 혜택을 받음

이 튜토리얼에서는 에피소드가 멀티 에이전트 상호작용의 전체 흐름을 어떻게 캡처하는지, 그리고 반영이 시간이 지남에 따라 헬스케어 어시스턴트를 개선하기 위한 실행 가능한 인사이트를 어떻게 제공하는지 확인할 수 있습니다.

---
## 시나리오 컨텍스트

다음으로 구성된 **헬스케어 어시스턴트 시스템**을 만들겠습니다:
1. 환자 질문을 라우팅하는 **슈퍼바이저 에이전트**
2. 보험 및 청구를 담당하는 **청구 에이전트**
3. 환자 정보를 담당하는 **인구통계 에이전트**
4. 처방전을 담당하는 **약물 에이전트**

모든 에이전트는 메모리 훅을 사용하여 대화를 에피소딕 메모리에 자동으로 저장합니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="75%" />
</div>

## 사전 요구사항

- Python 3.10+
- Bedrock 및 AgentCore 메모리 권한이 있는 AWS 자격 증명
- Amazon HealthLake 데이터스토어 (선택 사항)

시작하겠습니다!

## 1단계: 환경 설정
이 노트북이 작동하는 데 필요한 모든 라이브러리를 가져오고 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
%pip install -qr ./requirements.txt

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from bedrock_agentcore.memory import MemoryClient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare-assistant")

메모리 구성을 위한 사용자 입력을 정의하겠습니다.

In [ ]:
MEMORY_NAME = "healthcare_episodic_memory"
PATIENT_ID = "b2055b4d-ac17-4d94-8c5b-3395e4c334dd"
region = "us-east-1"  # 사용하는 AWS 리전으로 교체하세요
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}"
MODEL_ID = (
    "global.anthropic.claude-sonnet-4-20250514-v1:0"  # 사용하는 모델 ID로 교체하세요
)

print("메모리 구성:")
print(f"  메모리 이름: {MEMORY_NAME}")
print(f"  환자 ID: {PATIENT_ID}")
print(f"  리전: {region}")
print(f"  세션 ID: {SESSION_ID}")
print(f"  모델 ID: {MODEL_ID}")

## 2단계: HealthLake 데이터스토어 구성

헬스케어 에이전트가 쿼리할 수 있도록 환자 데이터가 포함된 HealthLake FHIR 데이터스토어를 설정합니다.

In [ ]:
import boto3
import requests
import time
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# HealthLake 구성
HEALTHLAKE_REGION = (
    input("HealthLake 리전을 입력하세요 (또는 Enter를 눌러 us-east-1 사용): ").strip()
    or "us-east-1"
)
DATASTORE_ID = input(
    "HealthLake 데이터스토어 ID를 입력하세요 (또는 Enter를 눌러 새로 생성): "
).strip()

healthlake_client = boto3.client("healthlake", region_name=HEALTHLAKE_REGION)

# 제공되지 않은 경우 새 데이터스토어 생성
if not DATASTORE_ID:
    create_new = (
        input(
            "\n데이터스토어 ID가 제공되지 않았습니다. Synthea 데이터로 새 HealthLake 데이터스토어를 생성하시겠습니까? (yes/no): "
        )
        .strip()
        .lower()
    )

    if create_new == "yes":
        print("\nHealthLake 데이터스토어 생성 중...")

        # 데이터스토어 생성
        create_response = healthlake_client.create_fhir_datastore(
            DatastoreName=f"healthcare-demo-{int(time.time())}",
            DatastoreTypeVersion="R4",
            PreloadDataConfig={"PreloadDataType": "SYNTHEA"},
        )

        DATASTORE_ID = create_response["DatastoreId"]
        print(f"데이터스토어 생성됨: {DATASTORE_ID}")
        print(
            "데이터스토어가 ACTIVE 상태가 될 때까지 대기 중 (10-15분 소요될 수 있습니다)..."
        )

        # ACTIVE 상태 대기
        while True:
            status_response = healthlake_client.describe_fhir_datastore(
                DatastoreId=DATASTORE_ID
            )
            status = status_response["DatastoreProperties"]["DatastoreStatus"]

            if status == "ACTIVE":
                print("데이터스토어가 ACTIVE 상태입니다")
                break
            elif status in ["FAILED", "DELETING"]:
                print(f"데이터스토어 생성 실패 상태: {status}")
                raise Exception(f"데이터스토어 생성 실패: {status}")

            print(f"   상태: {status}...")
            time.sleep(30)

        print(f"\nSynthea 데이터 로드 완료. 기본 환자 ID 사용: {PATIENT_ID}")


# HealthLake 엔드포인트 가져오기
datastore = healthlake_client.describe_fhir_datastore(DatastoreId=DATASTORE_ID)
HEALTHLAKE_ENDPOINT = datastore["DatastoreProperties"]["DatastoreEndpoint"]


def query_healthlake(resource_type, search_params=None, resource_id=None):
    """HealthLake FHIR API 쿼리"""
    if resource_id:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}/{resource_id}"
    else:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}"
        if search_params:
            params = "&".join([f"{k}={v}" for k, v in search_params.items()])
            url += f"?{params}"

    session = boto3.Session()
    credentials = session.get_credentials()

    request = AWSRequest(
        method="GET", url=url, headers={"Accept": "application/fhir+json"}
    )
    SigV4Auth(credentials, "healthlake", HEALTHLAKE_REGION).add_auth(request)

    response = requests.get(url, headers=dict(request.headers))

    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"가져오기 실패: {response.text}"}


print(f"\n{'=' * 70}")
print("HealthLake 구성:")
print(f"  데이터스토어 ID: {DATASTORE_ID}")
print(f"  엔드포인트:     {HEALTHLAKE_ENDPOINT}")
print(f"  리전:           {HEALTHLAKE_REGION}")
print(f"  환자 ID:        {PATIENT_ID}")
print(f"{'=' * 70}")

## 3단계: 에피소딕 전략으로 메모리 생성

여러 브랜치를 지원하는 단일 메모리 리소스를 생성합니다 - 각 에이전트마다 하나씩. 이 공유 메모리 리소스는 기반 역할을 하고, 브랜치는 각 에이전트의 대화에 대해 격리된 컨텍스트를 제공합니다.

Git 저장소와 비슷하게 생각하면 됩니다: 하나의 저장소(메모리 리소스)에 여러 브랜치(에이전트 컨텍스트)가 있는 구조입니다.

In [ ]:
client = MemoryClient(region_name=region)

strategies = [
    {
        "episodicMemoryStrategy": {
            "name": "HealthcareEpisodes",
            "description": "Captures healthcare interactions as episodes",
            "namespaces": ["healthcare/{actorId}/{sessionId}/"],
            "reflectionConfiguration": {"namespaces": ["healthcare/{actorId}/"]},
        }
    }
]

try:
    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        strategies=strategies,
        description="Healthcare system with episodic memory",
        event_expiry_days=7,  # 단기 대화는 7일 후 만료
        max_wait=300,
        poll_interval=10,
    )
    memory_id = memory["id"]
    logger.info(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(
        e
    ):
        # 메모리가 이미 존재하면 해당 ID를 가져옴
        memories = client.list_memories()
        memory_id = next(
            (m["id"] for m in memories if m["id"].startswith(MEMORY_NAME)), None
        )
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    print(f"오류: {e}")
    import traceback

    traceback.print_exc()

    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 헬스케어 멀티 에이전트 시스템을 위한 메모리 브랜칭 이해

생성한 메모리 리소스는 **브랜칭**을 지원합니다 - 이는 헬스케어 멀티 에이전트 아키텍처에 중요한 기능입니다. 작동 방식은 다음과 같습니다:

**단일 메모리 리소스, 다중 브랜치:**
- 모든 에이전트가 동일한 `memory_id`와 `session_id`를 공유
- 각 에이전트는 격리된 컨텍스트를 위한 자체 `branch_name`을 가짐

**헬스케어 멀티 에이전트 시스템의 주요 이점:**

1. **컨텍스트 격리**: 각 에이전트는 간섭 없이 자체 대화 기록을 유지
   - 청구 에이전트는 보험 및 청구 대화만 확인
   - 인구통계 에이전트는 환자 정보 대화만 확인
   - 약물 에이전트는 처방 관련 대화만 확인
   - 슈퍼바이저는 주요 라우팅 및 조율 흐름을 확인

2. **병렬 실행 안전성**: 여러 에이전트가 동시에 실행 가능
   - 에이전트가 병렬로 실행될 때 메모리 충돌 없음
   - 각 브랜치는 독립적으로 접근 가능
   - 동시 처리가 필요한 헬스케어 워크플로우에 필수적

3. **명확한 감사 추적**: 각 에이전트의 상호작용 추적 가능
   - 각 헬스케어 에이전트가 논의한 내용 검사
   - 에이전트별 문제 디버깅
   - 환자 케어 대화 흐름 이해
   - 규정 준수 및 문서화 요구사항 유지

**헬스케어 브랜치 구조:**
- `main` 브랜치: 슈퍼바이저 라우팅 결정
- `claims_agent` 브랜치: 보험 및 청구 대화
- `demographics_agent` 브랜치: 환자 정보 업데이트
- `medication_agent` 브랜치: 처방 논의

## 4단계: 브랜치 지원이 포함된 메모리 훅 프로바이더 생성

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from strands.hooks import AgentInitializedEvent, MessageAddedEvent
from bedrock_agentcore.memory import MemorySessionManager


class HealthcareMemoryHooks(HookProvider):
    def __init__(
        self, memory_id: str, region_name: str = None, branch_name: str = "main"
    ):
        """MemorySessionManager로 훅을 초기화합니다.

        Args:
            memory_id: AgentCore 메모리 ID
            region_name: 메모리 서비스의 AWS 리전
            branch_name: 이 에이전트의 메모리 브랜치 이름 (기본값: "main")
        """
        if region_name is None:
            region_name = region  # 전역 리전 변수 사용

        self.memory_manager = MemorySessionManager(
            memory_id=memory_id, region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # 액터/세션 조합별 세션 객체 캐시
        self._branch_initialized = False  # 브랜치 생성 여부 추적

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """주어진 액터/세션에 대한 MemorySession을 가져오거나 생성합니다."""
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id, session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """main 브랜치가 아닌 경우 브랜치가 존재하지 않으면 초기화합니다."""
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 브랜치가 이미 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # 포크할 main 브랜치의 마지막 이벤트 가져오기
                main_events = memory_session.list_events(branch_name="main")
                if not main_events:
                    # main 브랜치에 초기 이벤트 생성
                    memory_session.add_turns(
                        [
                            ConversationalMessage(
                                "Healthcare system initialized", MessageRole.ASSISTANT
                            )
                        ]
                    )
                    main_events = memory_session.list_events(branch_name="main")

                if main_events:
                    last_event = main_events[-1]
                    # 브랜치 생성
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(
                                f"Starting {self.branch_name} healthcare branch",
                                MessageRole.ASSISTANT,
                            )
                        ],
                    )
                    logger.info(f"헬스케어 브랜치 생성됨: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(
                f"헬스케어 브랜치 {self.branch_name} 초기화 실패: {e}"
            )

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """헬스케어 에이전트 시작 시 최근 대화 기록 로드"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning(
                    "헬스케어 에이전트 상태에 actor_id 또는 session_id가 없습니다"
                )
                return

            # 필요한 경우 브랜치 초기화 (non-main 브랜치)
            if self.branch_name != "main":
                self._initialize_branch(actor_id, session_id)

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 이 브랜치에서 최근 5개 대화 턴 가져오기
            recent_turns = memory_session.get_last_k_turns(
                k=5, branch_name=self.branch_name, include_parent_branches=False
            )

            if recent_turns:
                # 에이전트의 시스템 프롬프트에 컨텍스트 추가
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get("role", "unknown").lower()
                        text = message.get("content", {}).get("text", "")
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages[-10:])  # 마지막 10개 메시지
                    event.agent.system_prompt += (
                        f"\n\nRecent healthcare conversation history:\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )
                    logger.info(
                        f"브랜치 '{self.branch_name}'에서 헬스케어 컨텍스트 로드 완료"
                    )

        except Exception as e:
            logger.error(f"헬스케어 대화 기록 로드 실패: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """적절한 브랜치에 헬스케어 대화 턴을 메모리에 저장"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning(
                    "헬스케어 에이전트 상태에 actor_id 또는 session_id가 없습니다"
                )
                return

            # 메모리 세션 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지 가져오기
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("빈 헬스케어 메시지 건너뜀")
                return

            # 역할 문자열을 MessageRole 열거형에 매핑
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 적절한 브랜치에 메시지 저장
            if self.branch_name == "main":
                # main 브랜치 - 일반적으로 턴 추가
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # non-main 브랜치 - 기존 브랜치에 추가
                # 브랜치가 없으면 초기화
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 기존 브랜치에 추가
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)],
                    branch={"name": self.branch_name},
                )

            logger.info(f"헬스케어 브랜치에 메모리 저장됨: {self.branch_name}")

        except Exception as e:
            logger.error(f"헬스케어 메시지 저장 실패: {e}")

    def get_session(self, actor_id: str, session_id: str):
        """직접 접근을 위한 메모리 세션 객체를 가져옵니다."""
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """훅 레지스트리에 헬스케어 메모리 훅을 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)


print("헬스케어 메모리 훅 프로바이더 정의 완료")

## 5단계: 메모리 브랜칭을 활용한 멀티 에이전트 헬스케어 아키텍처 생성

이 섹션에서는 **서로 다른 메모리 브랜치**를 사용하는 전문 헬스케어 에이전트를 생성하여 브랜칭 기능을 시연합니다:

### 헬스케어 브랜칭 전략:
- **Main 브랜치**: 슈퍼바이저의 라우팅 결정을 저장하고 기본 대화 스레드 역할
- **claims_agent 브랜치**: 보험, 청구 및 청구 대화를 위한 별도 브랜치
- **demographics_agent 브랜치**: 환자 정보 및 연락처 세부정보를 위한 별도 브랜치
- **medication_agent 브랜치**: 처방 및 약물 대화를 위한 별도 브랜치

각 전문 헬스케어 에이전트는 자체 브랜치에서 작동하며, 처음 사용될 때 main 대화에서 자동으로 포크됩니다. 이를 통해:

- 서로 다른 헬스케어 전문 분야에 대한 독립적인 대화 흐름
- 도메인별 의료 컨텍스트의 격리
- main 슈퍼바이저 대화 스레드의 보존
- 헬스케어 데이터 분리 요구사항 준수
- 다양한 유형의 환자 상호작용에 대한 명확한 감사 추적

### 헬스케어 에이전트 역할:
- **슈퍼바이저 에이전트**: 환자 질문을 적절한 전문가에게 라우팅
- **청구 에이전트**: 보험 청구, 청구 문의 및 보장 범위 질문 처리
- **인구통계 에이전트**: 환자 인구통계 정보 및 연락처 업데이트 관리
- **약물 에이전트**: 처방 질문, 용량 정보 및 약물 관리 처리

이 아키텍처는 민감한 헬스케어 대화가 적절히 격리되면서도 일관된 환자 케어 경험을 유지하도록 보장합니다.

### 브랜치된 메모리로 에이전트 생성

다음으로, 서로 다른 메모리 브랜치를 사용하는 에이전트의 시스템 프롬프트를 정의하고 생성합니다. 동일한 `actor_id`와 `session_id`를 사용하면서 서로 다른 `branch_name` 값을 사용하여 격리된 대화 컨텍스트를 만드는 방식에 주목하세요:

In [ ]:
# 헬스케어 슈퍼바이저의 시스템 프롬프트
SUPERVISOR_PROMPT = """You are a healthcare supervisor agent. Route patient questions to:
    - Claims Agent: for insurance, billing, claims questions
    - Demographics Agent: for personal info, contact details  
    - Medication Agent: for prescriptions, medications, dosage
    
    Respond briefly and indicate which agent you're routing to."""

# 청구 전문가의 시스템 프롬프트
CLAIMS_PROMPT = """You handle insurance claims. Use the get_patient_claims tool to fetch 
    current claim data from HealthLake. Answer questions about claims, billing, and coverage."""

# 인구통계 전문가의 시스템 프롬프트
DEMOGRAPHICS_PROMPT = """You handle patient demographics. Use the get_patient_demographics tool to 
    fetch current patient data from HealthLake. Answer questions about contact details and personal information."""

# 약물 전문가의 시스템 프롬프트
MEDICATION_PROMPT = """You handle medications. Use the get_patient_medications tool to fetch 
    current medication data from HealthLake. Answer questions about prescriptions and dosages."""

## 6단계: 헬스케어 도구 및 메모리 훅 설정

먼저 전문 헬스케어 에이전트를 위한 HealthLake 데이터 도구와 메모리 훅을 생성합니다. 각 에이전트는 특정 브랜치 이름으로 구성된 자체 메모리 훅을 갖습니다:
- 청구 에이전트는 `claims_agent` 브랜치 사용
- 인구통계 에이전트는 `demographics_agent` 브랜치 사용
- 약물 에이전트는 `medication_agent` 브랜치 사용
- 슈퍼바이저 에이전트는 `main` 브랜치 사용

이러한 헬스케어 에이전트가 호출될 때:
1. 훅이 브랜치가 존재하는지 확인
2. 존재하지 않으면 main 대화에서 새 브랜치를 포크
3. 에이전트의 헬스케어 대화가 전용 브랜치에 저장
4. 각 에이전트는 환자 데이터 프라이버시를 위한 격리된 컨텍스트 유지
5. HealthLake 데이터 도구가 실시간 환자 정보 접근 제공
6. 적절한 데이터 격리를 통한 헬스케어 규정 준수 유지

**HealthLake 데이터 도구:**
- `get_patient_claims` - 보험 청구 및 청구 정보 검색
- `get_patient_demographics` - 환자 연락처 및 인구통계 데이터 가져오기
- `get_patient_medications` - 현재 처방전 및 약물 데이터 가져오기

**메모리 브랜치 구조:**
- 각 에이전트는 자체 메모리 컨텍스트로 독립적으로 작동
- 환자 대화는 헬스케어 도메인별로 적절히 격리
- 슈퍼바이저가 분리를 유지하면서 에이전트 간 조율


In [ ]:
# 헬스케어 메모리 훅을 None으로 초기화
supervisor_hooks = None
claims_hooks = None
demographics_hooks = None
medication_hooks = None

In [ ]:
@tool
def get_patient_claims(patient_id: str = PATIENT_ID) -> dict:
    """Get patient insurance claims from HealthLake"""
    return query_healthlake("Claim", {"patient": patient_id})


@tool
def get_patient_medications(patient_id: str = PATIENT_ID) -> dict:
    """Get patient medications from HealthLake"""
    return query_healthlake("MedicationRequest", {"patient": patient_id})


@tool
def get_patient_demographics(patient_id: str = PATIENT_ID) -> dict:
    """Get patient demographic information from HealthLake"""
    return query_healthlake("Patient", resource_id=patient_id)


# 각 에이전트에 대한 메모리 훅 생성
supervisor_hooks = HealthcareMemoryHooks(memory_id, region, "main")
claims_hooks = HealthcareMemoryHooks(memory_id, region, "claims_agent")
demographics_hooks = HealthcareMemoryHooks(memory_id, region, "demographics_agent")
medication_hooks = HealthcareMemoryHooks(memory_id, region, "medication_agent")

print("HealthLake 도구 및 메모리 훅 생성 완료")

### 헬스케어 에이전트 생성

이제 설정한 도구와 메모리 훅을 사용하여 헬스케어 에이전트를 생성합니다:

- **슈퍼바이저 에이전트**: 질문 라우팅 (main 브랜치)
- **청구 에이전트**: 보험 및 청구 (claims_agent 브랜치)
- **인구통계 에이전트**: 환자 정보 (demographics_agent 브랜치)
- **약물 에이전트**: 처방전 (medication_agent 브랜치)

각 에이전트는 전문 시스템 프롬프트, 관련 도구, 격리된 대화를 위한 메모리 훅을 갖습니다.

In [ ]:
# 메모리 브랜칭이 포함된 전문 헬스케어 에이전트 생성
supervisor = Agent(
    model=MODEL_ID,
    system_prompt=SUPERVISOR_PROMPT,
    hooks=[supervisor_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

claims_agent = Agent(
    model=MODEL_ID,
    system_prompt=CLAIMS_PROMPT,
    tools=[get_patient_claims],
    hooks=[claims_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

demographics_agent = Agent(
    model=MODEL_ID,
    system_prompt=DEMOGRAPHICS_PROMPT,
    tools=[get_patient_demographics],
    hooks=[demographics_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

medication_agent = Agent(
    model=MODEL_ID,
    system_prompt=MEDICATION_PROMPT,
    tools=[get_patient_medications],
    hooks=[medication_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

print("HealthLake 도구 및 메모리 브랜칭이 포함된 헬스케어 에이전트 생성 완료")

#### 에피소딕 메모리가 포함된 헬스케어 멀티 에이전트 시스템이 준비되었습니다!!

## 헬스케어 어시스턴트를 테스트해 보겠습니다.

환자 케어 시나리오로 헬스케어 멀티 에이전트 시스템을 테스트합니다:

**시도해 볼 수 있는 예제 질문:**
- "What's the status of my insurance claims?" (보험 청구 상태가 어떻게 되나요?)
- "Can you update my contact information?" (연락처 정보를 업데이트할 수 있나요?)
- "What medications am I currently taking?" (현재 복용 중인 약물이 무엇인가요?)
- "Do I have any pending billing issues?" (미결 청구 문제가 있나요?)
- "What's my current address on file?" (파일에 있는 현재 주소가 무엇인가요?)
- "Are there any drug interactions with my prescriptions?" (처방전에 약물 상호작용이 있나요?)
- "How much do I owe for my recent visit?" (최근 방문에 대해 얼마를 지불해야 하나요?)
- "Can you tell me about my coverage details?" (보장 범위에 대해 알려주실 수 있나요?)

In [ ]:
# 헬스케어 에이전트와 대화형 채팅
print("헬스케어 어시스턴트 - 종료하려면 'quit'을 입력하세요\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        break

    if not user_input:
        continue

    # 슈퍼바이저가 라우팅 처리
    routing = str(supervisor(user_input))
    print(f"\n슈퍼바이저: {routing}")

    # 슈퍼바이저의 결정에 따라 적절한 에이전트로 라우팅
    if "claims agent" in routing.lower():
        response = str(claims_agent(user_input))
        print(f"\n청구 에이전트: {response}\n")
    elif "demographics agent" in routing.lower():
        response = str(demographics_agent(user_input))
        print(f"\n인구통계 에이전트: {response}\n")
    elif "medication agent" in routing.lower():
        response = str(medication_agent(user_input))
        print(f"\n약물 에이전트: {response}\n")

## 헬스케어 메모리 브랜치 검사

AgentCore 메모리 브랜칭의 주요 장점 중 하나는 각 헬스케어 에이전트의 대화 기록을 독립적으로 검사할 수 있다는 것입니다. 이는 다음에 중요합니다:

**헬스케어 멀티 에이전트 시스템 디버깅:**
- 각 헬스케어 에이전트가 환자와 논의한 내용을 정확히 확인
- 어떤 에이전트가 어떤 의료 문의를 처리했는지 식별
- 헬스케어 시스템을 통한 환자 정보 흐름 추적

**헬스케어 에이전트 조율 이해:**
- 에이전트가 별도의 의료 컨텍스트를 유지했는지 확인
- 동시 실행 중 환자 데이터 충돌이 발생하지 않았는지 확인
- 헬스케어 에이전트 상호작용의 타임라인 감사
- 격리된 대화 추적을 통한 HIPAA 준수 보장

**헬스케어별 이점:**
- **청구 에이전트**: 모든 보험 및 청구 논의 추적
- **인구통계 에이전트**: 환자 정보 업데이트 및 변경 모니터링
- **약물 에이전트**: 모든 처방 및 약물 대화 감사
- **슈퍼바이저 에이전트**: 라우팅 결정 및 환자 분류 검토

환자 상담 중 생성된 헬스케어 브랜치를 살펴보겠습니다:

In [ ]:
print("\n=== 헬스케어 메모리 브랜치 보기 ===")

if claims_hooks or demographics_hooks or medication_hooks:
    # 브랜치를 나열하기 위해 아무 메모리 세션이나 가져오기 (모두 같은 세션을 가리킴)
    hook = (
        claims_hooks
        if claims_hooks
        else (demographics_hooks if demographics_hooks else medication_hooks)
    )
    if hook:
        memory_session = hook.get_session(actor_id=PATIENT_ID, session_id=SESSION_ID)

        # 세션의 모든 브랜치 나열
        branches = memory_session.list_branches()
        print(f"\n세션에 총 {len(branches)}개의 브랜치가 있습니다:")
        for branch in branches:
            events = memory_session.list_events(branch_name=branch.name)
            print(f"  - 브랜치: {branch.name}")
            print(f"    - 이벤트: {len(events)}")
            print(f"    - 생성일: {branch.created}")

            # 이 브랜치의 최근 대화 출력
            if events:
                print("    - 최근 대화:")
                for event in events[-100:]:  # 마지막 10개 이벤트 표시
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"        {role}: {text[:500]}...")

        print("\n각 브랜치는 다른 에이전트의 메모리를 나타냅니다:")
        print("  * 'main' = 슈퍼바이저 에이전트 대화")
        print("  * 'claims_agent' = 청구 어시스턴트 대화")
        print("  * 'demographics_agent' = 인구통계 어시스턴트 대화")
        print("  * 'medication_agent' = 약물 어시스턴트 대화")
else:
    print(
        "메모리 훅을 찾을 수 없습니다. 먼저 훅을 생성하는 셀을 실행했는지 확인하세요."
    )

## 장기 헬스케어 메모리 검증: 에피소드 및 반영

**EpisodicStrategy**를 사용하여 헬스케어 시스템이 단기 대화를 구조화된 장기 환자 인사이트로 어떻게 변환했는지 살펴보겠습니다.

### 헬스케어 에피소드
에피소드는 통합된 환자 상호작용을 다음과 함께 캡처합니다:
- **임상 컨텍스트**: 환자 케어 목표 및 결과
- **에이전트 조율**: 슈퍼바이저와 전문 에이전트가 어떻게 함께 작동했는지
- **데이터 통합**: HealthLake 정보 검색 및 표시

### 환자 반영
반영은 다음에 대한 교차 에피소드 인사이트를 제공합니다:
- **케어 패턴**: 환자 커뮤니케이션 선호도 및 반복적인 필요사항
- **효과적인 전략**: 이 환자에게 가장 잘 작동하는 접근 방식
- **최적화 기회**: 향후 케어 개선 영역

에피소드와 반영은 비동기적으로 처리됩니다.

In [ ]:
print("=== 헬스케어 장기 메모리: 에피소드 & 반영 ===")
actor_id = PATIENT_ID
session_id = SESSION_ID
# 헬스케어 에피소드 및 반영을 위한 네임스페이스 정의
episode_namespace = f"healthcare/{actor_id}/{session_id}/"
reflection_namespace = f"healthcare/{actor_id}/"
print(f"\n에피소드 네임스페이스: {episode_namespace}")
print(f"반영 네임스페이스: {reflection_namespace}")

try:
    print("\n헬스케어 에피소드 (세션별 환자 상호작용)")
    episodes = client.retrieve_memories(
        memory_id=memory_id,
        namespace=episode_namespace,
        query="patient healthcare interactions",
        top_k=10,
    )
    print(f"{len(episodes)}개의 헬스케어 에피소드 발견")

    for i, episode in enumerate(episodes, 1):
        print(f"\n헬스케어 에피소드 {i}:")
        content = episode.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # 헬스케어 컨텍스트를 위해 더 많은 내용 표시
            print(f"   {text[:300]}..." if len(text) > 300 else f"   {text}")
        print(f"   점수: {episode.get('score', 'N/A')}")

    if not episodes:
        print("   아직 에피소드가 발견되지 않았습니다. 에피소딕 처리는 비동기적으로 수행됩니다.")

except Exception as e:
    print(f"헬스케어 에피소드 검색 오류: {e}")

try:
    print("\n환자 반영 (교차 세션 헬스케어 인사이트)")
    reflections = client.retrieve_memories(
        memory_id=memory_id,
        namespace=reflection_namespace,
        query="patient care patterns and insights",
        top_k=10,
    )
    print(f"{len(reflections)}개의 환자 반영 발견")

    for i, reflection in enumerate(reflections, 1):
        print(f"\n환자 반영 {i}:")
        content = reflection.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # 헬스케어 인사이트를 위해 더 많은 내용 표시
            print(f"   {text[:400]}..." if len(text) > 400 else f"   {text}")
        print(f"   점수: {reflection.get('score', 'N/A')}")

    if not reflections:
        print(
            "   아직 반영이 발견되지 않았습니다. 반영은 여러 에피소드 이후에 생성됩니다."
        )

except Exception as e:
    print(f"환자 반영 검색 오류: {e}")

print(
    "\n팁: 대화형 헬스케어 메모리 시각화를 위해 메모리 브라우저를 사용하세요"
)
print("   에피소드는 개별 환자 상담 요약을 보여줍니다")
print("   반영은 환자 케어 및 선호도의 패턴을 보여줍니다")
print(
    "\n참고: 에피소드 및 반영 생성은 대화 후 10-15분이 소요됩니다"
)
print("   에피소드/반영이 즉시 나타나지 않으면 나중에 다시 확인하세요")

## 요약

### 구축한 내용:
1. **슈퍼바이저 에이전트** - main 브랜치에서 오케스트레이션
2. **청구 에이전트** - claims_agent 브랜치에서 보험 처리
3. **인구통계 에이전트** - demographics_agent 브랜치에서 환자 정보 관리
4. **약물 에이전트** - medication_agent 브랜치에서 약물 처리

### 메모리 아키텍처:
- **단기**: 각 에이전트는 격리된 브랜치를 가짐
- **에피소드**: 세션별 저장 `healthcare/{actorId}/{sessionId}/`
- **반영**: 모든 세션에서 공유 `healthcare/{actorId}/`

### 이점:
- 에이전트가 서로의 대화를 간섭하지 않음
- 모든 에이전트가 동일 세션의 장기 메모리에 기여
- 학습된 패턴(반영)이 모든 환자 세션에서 공유
- 에이전트별 완전한 대화 기록 유지

## 정리 (선택 사항)

이 튜토리얼에서 생성한 메모리와 IAM 역할을 삭제하려면 이 셀을 실행하세요.

In [ ]:
# import boto3

# print("Cleanup Options:")
# delete_memory = input("Delete memory? (yes/no): ").strip().lower()
# if delete_memory == 'yes':
#   try:
#       print(f"Deleting memory: {memory_id}")
#       client.delete_memory_and_wait(memory_id=memory_id)
#       print("Memory deleted")
#   except Exception as e:
#       print(f"Error deleting memory: {e}")
# else:
#   print(f"Memory preserved: {memory_id}")

# delete_healthlake = input("Delete HealthLake datastore? (yes/no): ").strip().lower()
# if delete_healthlake == 'yes':
#     try:
#         print(f"Deleting HealthLake datastore: {DATASTORE_ID}")
#         healthlake_client.delete_fhir_datastore(DatastoreId=DATASTORE_ID)
#         print("HealthLake datastore deletion initiated")
#     except Exception as e:
#         print(f"Error deleting HealthLake datastore: {e}")
# else:
#     print(f"HealthLake datastore preserved: {DATASTORE_ID}")

# print("Cleanup complete")